# 5. 提出ファイルを作る

04 が出した446人分の生存確率を、投稿できる形に整える。

やることは形式を合わせるだけ。ただしここを間違えると、
**中身が正しくても点数が出ない**。確認しながら進める。

In [ ]:
import pandas as pd

# 04 が保存した予測
pred = pd.read_csv("../data/processed/pred_test.csv", index_col=0)

# 提出フォーマットの見本。header=None は「1 行目は列名ではなくデータ」の指定
sample = pd.read_csv("../data/raw/sample_submit.csv", index_col=0, header=None)

print("pred:", pred.shape, " sample:", sample.shape)

## 1. 提出の形を確かめる

見本のファイルを開くと、こうなっている。

```
0,0
1,1
2,0
```

決まりが3つ読み取れる。

- **1行目から中身が始まる**（項目名の行がない）
- **1列目が id** — 評価用データの id と同じであること
- **2列目が予測**

見本の2列目は `0` と `1` だが、これは見本の値。
決まりの上では**生存確率（0.0〜1.0 の小数）**を書く。

In [ ]:
# 見本を読み込むと、列名が 1、index 名が 0 になる
#   ヘッダが無いので、pandas が位置の番号をそのまま名前にしている
print("列名:", list(sample.columns), " index 名:", sample.index.name)
print("見本に入っている値:", sorted(sample[1].unique()))
print()
print("id が pred と完全一致:", list(sample.index) == list(pred.index))

# 出力の見方
#   id の一致は必ず確認する。ここがずれていると、別人の予測を出すことになる

## 2. 確率のまま出す

この課題の採点には **AUC** という方法が使われる。
名前は Area Under the Curve（曲線の下の面積）の略だが、
実際にやっているのは「**順番が正しく並んでいるか**」の測定。

4人で考えてみる。助かった2人と助からなかった2人を、確率の高い順に並べる。

```
確率    実際
0.969   助かった ○
0.626   死亡     ×    ← この人が上に来てしまった
0.528   助かった ○
0.121   死亡     ×
```

理想は、助かった人が全員上に来ること。
そこで「助かった人」と「助からなかった人」を1人ずつ組にして、
**助かった人の方が上にいるか**を数える。組み合わせは 2 × 2 = 4通り。

| 助かった人 | 死亡した人 | 助かった人の方が上か |
| --- | --- | --- |
| 0.969 | 0.626 | ○ |
| 0.969 | 0.121 | ○ |
| 0.528 | 0.626 | **×** |
| 0.528 | 0.121 | ○ |

4組のうち3組が正しい順番なので、AUC は 3 ÷ 4 = **0.75**。

| AUC | 意味 |
| --- | --- |
| 1.0 | 助かった人が全員上。完璧 |
| 0.5 | でたらめ。全員に同じ値を出すとこうなる |

**順番しか見ていない**ので、`0` と `1` に丸めてしまうと損をする。
同じ値だらけになって順番が付かなくなるため。実際に比べてみる。

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

# 04 と同じ手順で検証用データを用意して、確率のまま出す場合と丸めた場合を比べる
train = pd.read_csv("../data/processed/train_processed.csv", index_col=0)
X = train.drop(columns=["survived"]); y = train["survived"]
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)
m = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
prob = m.predict_proba(X_valid)[:, 1]

print("確率のまま  :", round(roc_auc_score(y_valid, prob), 4))
print("0/1 に丸める:", round(roc_auc_score(y_valid, (prob > 0.5).astype(int)), 4))

# 出力の見方
#   丸めると 0.81 から 0.73 まで落ちる。同じモデルなのに、出し方だけで 8 ポイント差
#   「この人は 48% くらい」という手持ちの情報を捨てているため
#
#   だから提出は確率のまま。0/1 には変換しない

## 3. 見本の値を置き換える

見本ファイルの2列目を、自分の予測で上書きする。
id が一致していることは1節で確認済みなので、そのまま使える。

In [ ]:
# sample[1] が予測値の列。ここを pred の中身に差し替える
#   .values で数値の並びだけを取り出している。
#   DataFrame のまま代入すると、列名の違いで噛み合わないことがある
sample[1] = pred["pred"].values

sample.head()

# 出力の見方
#   2 列目が 0/1 から小数に変わっている
#   左端の id は見本のものをそのまま使っているので、順序も仕様どおり

## 4. 項目名の行を付けずに書き出す

`to_csv` は何も指定しないと、1行目に項目名を書いてしまう。
提出の決まりに合わせて、書かないように指定する。

In [ ]:
# header=False で列名の行を書かない
#   index=True で id を残す（既定なので省略してもよいが、意図として明示する）
sample.to_csv("../data/submissions/submit.csv", header=False, index=True)

print("書き出した")

In [ ]:
# 書き出したファイルをそのまま読み直して、仕様どおりか確認する
#   自分が持っている変数ではなくファイルを見るのが要点。
#   書き出す処理そのものが間違っていた場合、変数を見ても気づけない
check = pd.read_csv("../data/submissions/submit.csv", header=None)

print("行数:", len(check), "（446 行であること）")
print("列数:", check.shape[1], "（2 列であること）")
print("id が評価用データと一致:", check[0].tolist() == list(pred.index))
print("予測値の範囲:", round(check[1].min(), 4), "〜", round(check[1].max(), 4), "（0〜1 に収まること）")
print("欠損:", int(check.isna().sum().sum()), "（0 であること）")

# ヘッダの有無は、ファイルの 1 行目を文字として見るのが確実
with open("../data/submissions/submit.csv") as f:
    print("ファイルの 1 行目:", f.readline().strip(), "（列名ではなくデータであること）")

## 5. 投稿する

`data/submissions/submit.csv` をコンペの投稿ページからアップロードする。
しばらくすると AUC での結果が通知される。

手元の検証用データでの AUC が 0.81 前後だったので、その付近が目安。
大きく下回った場合は、モデルを疑う前に**提出の形が崩れていないか**を確認する。

## この章のまとめ

- 提出の形は**項目名の行なし**・1列目が id・2列目が予測
- 採点方法が AUC なので、**確率のまま出す**。
  `0` と `1` に丸めると、検証用データでは 0.81 から 0.73 まで落ちた
- 見本ファイルの id をそのまま使い、2列目だけを差し替えた。
  id を自分で作り直すと、並び順の取り違えで別人の予測を出してしまう危険がある
- 書き出した後は**ファイルを読み直して**、行数・列数・1行目・id・値の範囲を確認した

ここまでで一通りの流れが終わり。精度を上げたい場合は、
`age` の扱いを変える、埋め方を見直す、別の仕組みを試す、といった方向になる。